# Tutorial 13 -- Spin Echo and Dephasing Mitigation

Compare a Ramsey-style free-evolution experiment to a Hahn-echo sequence under the same dephasing model and fit the echo envelope.

**Prerequisites.** Tutorials 11 and 12 are recommended first.


## 1. Goal

We will show that a spin-echo sequence suppresses static phase accumulation and extends the visible coherence envelope relative to a Ramsey-style experiment.


## 2. Physical Background

A Hahn echo inserts a `pi` pulse halfway through the free evolution. That pulse reverses the sign of static phase accumulation, which makes the sequence less sensitive to low-frequency detuning offsets than Ramsey.


## 3. Imports


In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from cqed_sim import (
    AmplifierChain,
    BosonicModeSpec,
    DispersiveCouplingSpec,
    DispersiveReadoutTransmonStorageModel,
    DispersiveTransmonCavityModel,
    DisplacementGate,
    FrameSpec,
    NoiseSpec,
    Pulse,
    PurcellFilter,
    QubitMeasurementSpec,
    ReadoutChain,
    ReadoutResonator,
    RotationGate,
    SidebandDriveSpec,
    SequenceCompiler,
    SimulationConfig,
    StatePreparationSpec,
    TransmonModeSpec,
    UniversalCQEDModel,
    build_displacement_pulse,
    build_rotation_pulse,
    build_sideband_pulse,
    carrier_for_transition_frequency,
    coherent_state,
    compute_energy_spectrum,
    fock_state,
    manifold_transition_frequency,
    measure_qubit,
    prepare_simulation,
    prepare_state,
    pure_dephasing_time_from_t1_t2,
    qubit_state,
    run_rabi,
    run_ramsey,
    run_spectroscopy,
    run_t1,
    run_t2_echo,
    sideband_transition_frequency,
    simulate_batch,
    simulate_sequence,
)
from cqed_sim.plotting import plot_energy_levels
from cqed_sim.pulses import gaussian_envelope, square_envelope
from cqed_sim.sim import (
    cavity_wigner,
    conditioned_bloch_xyz,
    mode_moments,
    qubit_conditioned_mode_moments,
    readout_response_by_qubit_state,
    reduced_cavity_state,
    reduced_qubit_state,
    reduced_storage_state,
    storage_photon_number,
    subsystem_level_population,
    transmon_level_populations,
)
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_ghz,
    angular_to_hz,
    angular_to_mhz,
    final_expectation,
    fit_echo_signal,
    fit_exponential_decay,
    fit_lorentzian_peak,
    fit_rabi_vs_amplitude,
    fit_rabi_vs_duration,
    fit_ramsey_signal,
    ns,
    us,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 4. Simulation Parameters


In [ ]:
t1_true = 40.0 * us
t2_star_true = 10.0 * us
t2_echo_target = 18.0 * us
delays_us = np.linspace(0.0, 24.0, 37)
rotation_duration = 30.0 * ns
rotation_sigma_fraction = 0.18
dt = 4.0 * ns


## 5. Model Construction


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.0),
    omega_q=GHz(6.2),
    alpha=0.0,
    chi=0.0,
    kerr=0.0,
    n_cav=1,
    n_tr=2,
)
frame = FrameSpec(omega_q_frame=model.omega_q)
tphi_true = pure_dephasing_time_from_t1_t2(t1_s=t1_true, t2_s=t2_star_true)
noise = NoiseSpec(t1=t1_true, tphi=tphi_true)
x90_pulses, _, _ = build_rotation_pulse(
    RotationGate(index=0, name="x90", theta=np.pi / 2.0, phi=0.0),
    {"duration_rotation_s": rotation_duration, "rotation_sigma_fraction": rotation_sigma_fraction},
)
x180_pulses, _, _ = build_rotation_pulse(
    RotationGate(index=1, name="x180", theta=np.pi, phi=0.0),
    {"duration_rotation_s": rotation_duration, "rotation_sigma_fraction": rotation_sigma_fraction},
)
x90 = x90_pulses[0]
x180 = x180_pulses[0]


## 6. Pulse / Sequence Construction


In [ ]:
delays_s = delays_us * us
ramsey_like = []
echo = []
for delay_s in delays_s:
    ramsey_pulses = [
        Pulse(x90.channel, 0.0, x90.duration, x90.envelope, amp=x90.amp, carrier=x90.carrier, phase=0.0, label="ramsey_a"),
        Pulse(x90.channel, x90.duration + delay_s, x90.duration, x90.envelope, amp=x90.amp, carrier=x90.carrier, phase=0.0, label="ramsey_b"),
    ]
    ramsey_t_end = 2.0 * x90.duration + delay_s + dt
    ramsey_compiled = SequenceCompiler(dt=dt).compile(ramsey_pulses, t_end=ramsey_t_end)
    ramsey_result = simulate_sequence(
        model,
        ramsey_compiled,
        model.basis_state(0, 0),
        {"qubit": "qubit"},
        config=SimulationConfig(frame=frame, max_step=dt),
        noise=noise,
    )
    ramsey_like.append(final_expectation(ramsey_result, "P_e"))

    echo_pulses = [
        Pulse(x90.channel, 0.0, x90.duration, x90.envelope, amp=x90.amp, carrier=x90.carrier, phase=0.0, label="echo_a"),
        Pulse(x180.channel, x90.duration + 0.5 * delay_s, x180.duration, x180.envelope, amp=x180.amp, carrier=x180.carrier, phase=0.0, label="echo_pi"),
        Pulse(x90.channel, x90.duration + delay_s + x180.duration, x90.duration, x90.envelope, amp=x90.amp, carrier=x90.carrier, phase=0.0, label="echo_b"),
    ]
    echo_t_end = 2.0 * x90.duration + x180.duration + delay_s + dt
    echo_compiled = SequenceCompiler(dt=dt).compile(echo_pulses, t_end=echo_t_end)
    echo_result = simulate_sequence(
        model,
        echo_compiled,
        model.basis_state(0, 0),
        {"qubit": "qubit"},
        config=SimulationConfig(frame=frame, max_step=dt),
        noise=noise,
    )
    echo.append(final_expectation(echo_result, "P_e"))
ramsey_like = np.asarray(ramsey_like, dtype=float)
echo = np.asarray(echo, dtype=float)


## 7. Running the Simulation


In [ ]:
fit = fit_echo_signal(delays_s, echo, p0=(t2_echo_target, 0.5, 0.5))
print(f"Representative echo-envelope fit T2_echo = {fit.parameters['t2_echo'] / us:.3f} us")


## 8. Visualizing the Results


In [ ]:
fig, ax = plt.subplots()
ax.plot(delays_us, ramsey_like, "o-", label="Ramsey-style sequence")
ax.plot(delays_us, echo, "o-", label="Spin echo")
ax.plot(delays_us, fit.model_y, "--", label="Echo fit")
ax.set_xlabel("Total delay [us]")
ax.set_ylabel(r"Final $P_e$")
ax.set_title("Spin echo mitigates low-frequency dephasing")
ax.legend()
plt.show()


## 9. Physical Interpretation

The echo trace decays more slowly because the middle `pi` pulse refocuses static or slowly varying phase errors. That is why echo is often used to distinguish reversible dephasing from irreversible coherence loss.


## 10. Exercises / Next Steps

- Add a deliberate detuning offset and verify that the Ramsey-like sequence is much more sensitive to it than the echo sequence.
- Compare the fitted echo envelope to the direct `run_t2_echo(...)` calibration-target helper introduced later in Tutorial 23.
- Continue to Tutorial 14 for bosonic Kerr dynamics.
